In [ ]:
# One notebook for the whole pair-progress setup: Left one-click script, what to upload to Databricks,
# the collector job (the "utils") and its schedule, the progress tables/views, and the AI/BI dashboard.
# Every cell writes files under OUT or prints the exact call; only EXECUTE=True talks to Databricks.
from pathlib import Path
import json, os, subprocess, sys, tempfile

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(ROOT / 'src'))
OUT = Path(os.getenv('DTRACK_PROGRESS_OUT', Path(tempfile.gettempdir()) / 'dtrack-progress'))   # generated files
EXECUTE = os.getenv('DTRACK_EXECUTE') == '1'                                   # False: dry run everywhere
SCHEMA = os.getenv('DTRACK_PROGRESS_SCHEMA', 'main.dtrack')                   # catalog.schema of the three tables
RUN_S3 = os.getenv('DTRACK_RUN_S3', 's3://my-bucket/dtrack/runs')             # Left writes here
RUN_ROOTS = [r for r in os.getenv('DTRACK_RUN_ROOTS', '/Volumes/main/dtrack/work/dtrack/v1/runs').split(',') if r]  # same objects, Volume view
WORKSPACE_DIR = os.getenv('DTRACK_WORKSPACE_DIR', '/Workspace/Shared/dtrack')  # where src/dtrack_right lands
WAREHOUSE_ID = os.getenv('DTRACK_DBX_WAREHOUSE_ID', '')
CLUSTER_ID = os.getenv('DTRACK_DBX_CLUSTER_ID', '')                            # empty = serverless job compute
PROFILE = os.getenv('DTRACK_DBX_PROFILE', 'corp')
CONFIG = Path(os.getenv('DTRACK_CONFIG', ROOT / 'examples/e2e/config.json'))

from dtrack_right import progress
OUT.mkdir(parents=True, exist_ok=True)
os.chdir(OUT)                                                                  # %%writefile cells below are relative to OUT
w = None
if EXECUTE:
    from dtrack_left.backends.databricks import workspace, upload_workspace, run_sql
    w = workspace(PROFILE)
{'out': str(OUT), 'execute': EXECUTE, 'schema': SCHEMA, 'run_roots': RUN_ROOTS, 'workspace_dir': WORKSPACE_DIR}

In [ ]:
%%writefile left.job.sh
#!/usr/bin/env bash
# dtrack Left one-click: doctor, Stage 1, Stage 2 for one run id. Rerun with the same RUN_ID to resume
# (windows whose units are all published are skipped). The Right job on Databricks drains the same run.
set -euo pipefail
CONFIG="${1:?usage: left.job.sh config.json [run_id]}"
RUN_ID="${2:-$(date -u +%Y%m%dT%H%M%SZ)-$(basename "$CONFIG" .json)}"
: "${DTRACK_RUN_S3:?set DTRACK_RUN_S3 (s3://.../runs, no run id)}"
export DTRACK_DUCKDB_MEMORY="${DTRACK_DUCKDB_MEMORY:-16GB}"   # process budget, split across pull + stat workers
export DTRACK_DUCKDB_SHARE="${DTRACK_DUCKDB_SHARE:-4}"
WORK="${DTRACK_COL_WORK_DIR:-.dtrack-work/col_check}"
dtrack tools doctor --config "$CONFIG"
dtrack row-check --config "$CONFIG" --run-s3 "$DTRACK_RUN_S3" --run-id "$RUN_ID" --workers "${WORKERS:-4}"
dtrack col-check --run-s3 "$DTRACK_RUN_S3" --run-id "$RUN_ID" --work-dir "$WORK" \
  --pull-workers "${PULL_WORKERS:-2}" --stat-workers "${STAT_WORKERS:-2}"
echo "run_id=$RUN_ID run=$DTRACK_RUN_S3/$RUN_ID"

In [ ]:
# Syntax-check the script; usage: DTRACK_RUN_S3=s3://.../runs bash left.job.sh config.json [run_id]
assert subprocess.run(['bash', '-n', 'left.job.sh'], capture_output=True, text=True).returncode == 0
Path('left.job.sh').chmod(0o755)
print(Path('left.job.sh').read_text())

In [ ]:
%%writefile progress_notebook.py
# Databricks notebook source
# dtrack progress collector. Reads every run below each root (read-only), writes one snapshot row per
# run x pair into <schema>.dtrack_progress_snapshot, then opens/closes AUTO blockers. Lives next to
# dtrack_right/ in the workspace, exactly like notebook.py (the Right job).

# COMMAND ----------

# MAGIC %pip install -q -r ./requirements.txt

# COMMAND ----------

dbutils.library.restartPython()

# COMMAND ----------

import os, sys
from datetime import UTC, datetime

sys.path.insert(0, os.path.dirname(os.getcwd()))
from dtrack_right import progress

dbutils.widgets.text("run_roots", "/Volumes/main/dtrack/work/dtrack/v1/runs")   # comma separated
dbutils.widgets.text("schema", "main.dtrack")
dbutils.widgets.text("keep_complete_days", "1")
roots = [r.strip() for r in dbutils.widgets.get("run_roots").split(",") if r.strip()]
schema = dbutils.widgets.get("schema")
now = datetime.now(UTC)

for statement in progress.ddl(schema) + progress.views(schema):   # idempotent
    spark.sql(statement)

rows = progress.collect(roots, now=now, keep_complete_days=float(dbutils.widgets.get("keep_complete_days")))
if rows:
    spark.sql(progress.insert_snapshot_sql(schema, rows))
    last = {(r.run_id, r.pair_name): r.last_progress_at.isoformat()
            for r in spark.sql(progress.last_progress_sql(schema)).collect()}
    merge = progress.merge_blockers_sql(schema, progress.auto_blockers(rows, now=now, last_progress_at=last))
    spark.sql(merge)
print({"runs": len({r["run_id"] for r in rows}), "rows": len(rows),
       "errors": sum(1 for r in rows if r["collector_error"])})

In [ ]:
# What goes to Databricks: the dtrack_right package (notebook.py inside it is the Right job) and the
# collector next to it. upload_workspace mirrors a folder; dry_run lists without writing.
uploads = [
    (ROOT / 'src/dtrack_right', f'{WORKSPACE_DIR}/dtrack_right'),
    (OUT / 'progress_notebook.py', f'{WORKSPACE_DIR}/dtrack_right/progress_notebook.py'),
]
files = sorted(str(p.relative_to(ROOT)) for p in (ROOT / 'src/dtrack_right').rglob('*.py') if '__pycache__' not in p.parts)
assert 'src/dtrack_right/notebook.py' in files and 'src/dtrack_right/progress.py' in files
if EXECUTE:
    for source, target in uploads:
        print(upload_workspace(w, source, target, overwrite=True))
{'uploads': [(str(s), t) for s, t in uploads], 'files': len(files)}

In [ ]:
# Tables and views. The collector also runs these (idempotent); here they go through the SQL warehouse so
# the dashboard can be built before the first collection. Statements are printed for review either way.
statements = progress.ddl(SCHEMA) + progress.views(SCHEMA)
for statement in statements:
    print(statement.split('\n')[0] + ' ...')
    if EXECUTE:
        run_sql(w, statement, WAREHOUSE_ID)
assert any('dtrack_progress_snapshot' in s for s in statements) and any('v_pair_status' in s for s in statements)
Path('progress_tables.sql').write_text(';\n\n'.join(statements) + ';\n')

In [ ]:
# Offline self-check of the collector on a fake run folder: one pair mid col-check with a dead unit and a
# stale heartbeat, one pair that has only finished its Left count.
from datetime import UTC, datetime, timedelta
from dtrack_right.backends.store import VolumeStore
from dtrack_right.common.protocol import left_row_path, plan_path, row_result_path, write_run_status

now = datetime.now(UTC)
runs = OUT / 'runs'
store = VolumeStore(runs / 'demo-run')
store.put_json(left_row_path('orders'), {'run_id': 'demo-run', 'pair': 'orders', 'counts': {'2026-01-01': 5}})
store.put_json(left_row_path('customers'), {'run_id': 'demo-run', 'pair': 'customers', 'counts': {}})
store.put_json('_status/left_row_check.closed.json', {'pairs': ['customers', 'orders'], 'closed_at': (now - timedelta(hours=3)).isoformat()})
store.put_json(row_result_path('orders'), {'run_id': 'demo-run', 'pair': 'orders',
    'summary': {'match': 3, 'count_diff': 1, 'left_only': 0, 'right_only': 0},
    'gate_decision': {'policy': 'auto', 'passed': True, 'excluded_ratio': 0.25}})
store.close_row_check(['orders'])
store.put_json(plan_path('orders'), {'pair': 'orders', 'expected_units': 4, 'windows': [], 'columns': []})
for seq, uid in enumerate(['u1', 'u2', 'u3'], start=1):
    store.commit_unit(seq, uid, {'n_total': seq}, {'pair': 'orders', 'window': '2026-01', 'column': f'c{seq}'})
store.put_result('u1', {'verdict': 'match'}); store.put_result('u2', {'verdict': 'diff'}); store.put_dead('u3', {'error': 'boom'})
write_run_status(store, {'state': 'right_col_check', 'role': 'right', 'run_id': 'demo-run'})
store.heartbeat((now - timedelta(minutes=45)).timestamp())

rows = progress.collect([str(runs)], now=now)
by_pair = {r['pair_name']: r for r in rows}
assert by_pair['orders']['units_compared'] == 2 and by_pair['orders']['units_dead'] == 1 and by_pair['orders']['units_diff'] == 1
assert by_pair['customers']['row_done_at'] is None and by_pair['customers']['row_left_done_at']
blockers = progress.auto_blockers(rows, now=now)
open_rules = sorted({b['category'] for b in blockers if b['resolved_at'] is None})
assert open_rules == ['DEAD_UNITS', 'ROW_COUNT_DIFF', 'STALE_HEARTBEAT'], open_rules
Path('demo_snapshot.sql').write_text(progress.insert_snapshot_sql(SCHEMA, rows) + ';\n\n' + progress.merge_blockers_sql(SCHEMA, blockers) + ';\n')
{'rows': len(rows), 'open_blockers': open_rules, 'sql': str(OUT / 'demo_snapshot.sql')}

In [ ]:
# Jobs: the Right job is started by hand per run (it exits by itself); the collector runs every 15 minutes.
# Empty CLUSTER_ID means serverless job compute; otherwise both tasks reuse that cluster.
def task(key, notebook, params):
    spec = {'task_key': key, 'notebook_task': {'notebook_path': f'{WORKSPACE_DIR}/dtrack_right/{notebook}', 'base_parameters': params}}
    if CLUSTER_ID:
        spec['existing_cluster_id'] = CLUSTER_ID
    return spec

jobs = {
    'dtrack-right': {
        'name': 'dtrack-right', 'max_concurrent_runs': 3, 'timeout_seconds': 0,
        'tasks': [task('right', 'notebook', {'run_s3': RUN_ROOTS[0], 'run_id': '', 'gate': 'auto'})],
    },
    'dtrack-progress-collector': {
        'name': 'dtrack-progress-collector', 'max_concurrent_runs': 1, 'timeout_seconds': 600,
        'schedule': {'quartz_cron_expression': '0 0/15 * * * ?', 'timezone_id': 'UTC', 'pause_status': 'UNPAUSED'},
        'tasks': [task('collect', 'progress_notebook', {'run_roots': ','.join(RUN_ROOTS), 'schema': SCHEMA})],
    },
}
Path('jobs.json').write_text(json.dumps(jobs, indent=2) + '\n')
created = {}
if EXECUTE:
    existing = {j.settings.name: j.job_id for j in w.jobs.list()}
    for name, spec in jobs.items():
        if name in existing:
            w.api_client.do('POST', '/api/2.2/jobs/reset', body={'job_id': existing[name], 'new_settings': spec})
            created[name] = existing[name]
        else:
            created[name] = w.api_client.do('POST', '/api/2.2/jobs/create', body=spec)['job_id']
assert jobs['dtrack-progress-collector']['schedule']['quartz_cron_expression'] == '0 0/15 * * * ?'
{'jobs': str(OUT / 'jobs.json'), 'created': created, 'start_right': 'dtrack tools dbx runs --active --job-id <id>'}

In [ ]:
%%writefile build_pair_progress_dashboard.py
# AI/BI dashboard (.lvdash.json) over v_pair_status / v_pair_daily / dtrack_pair_blocker: an Executive page
# and a Detailed page. Same widget helpers as the count-watch dashboard builder.
import argparse, json, re
from pathlib import Path

RED, ORANGE, BLUE, GREEN, GRAY = "red", "orange", "blue", "green", "gray"
TINT = {RED: ("#FDE2E1", "#A4161A"), ORANGE: ("#FFE8CC", "#8A4B00"), BLUE: ("#DCEEF6", "#075A73"),
        GREEN: ("#D4EDDA", "#155724"), GRAY: ("#ECEEF0", "#4A5560")}
SOLID = {RED: "#E5484D", ORANGE: "#F76B15", BLUE: "#0B7FA8", GREEN: "#30A46C", GRAY: "#9BA1A6"}
STATUSES = [("FAILED", "🔴", RED), ("BLOCKED", "🔴", RED), ("STALLED", "🟠", ORANGE), ("ROW_CHECK", "🔵", BLUE),
            ("COL_CHECK", "🔵", BLUE), ("REPORTING", "🔵", BLUE), ("NOT_STARTED", "⚪", GRAY), ("DONE", "🟢", GREEN)]
HEALTH = {"LATE": RED, "AT_RISK": ORANGE, "ON_TRACK": GREEN, "NO_TARGET": GRAY}
STATUS_TINTS = {s: c for s, _, c in STATUSES}
HOURS_SQL = "CASE WHEN {h} IS NULL THEN 'unknown' WHEN {h} < 24 THEN concat({h}, 'h') ELSE concat(floor({h} / 24), 'd ', {h} % 24, 'h') END"
AGE_SQL = "CASE WHEN {m} IS NULL THEN 'never' WHEN {m} < 60 THEN concat({m}, 'm ago') WHEN {m} < 1440 THEN concat(floor({m} / 60), 'h ', {m} % 60, 'm ago') ELSE concat(floor({m} / 1440), 'd ', floor(({m} % 1440) / 60), 'h ago') END"


def case(column, pairs, default):
    whens = "\n".join(f"    WHEN '{value}' THEN '{label}'" for value, label in pairs)
    return f"CASE {column}\n{whens}\n    ELSE {default}\n  END"


def dataset_sql(schema):
    label = case("status", [(s, f"{i} {s.replace('_', ' ').title()}") for s, i, _ in STATUSES], "status")
    severity = case("status", [(s, str(n)) for n, (s, _, _) in enumerate(STATUSES)], str(len(STATUSES)))
    status = f"""SELECT *,
  {label} AS status_label,
  CAST({severity} AS INT) AS severity,
  {HOURS_SQL.format(h="elapsed_hours")} AS elapsed,
  CASE WHEN eta_ts IS NULL THEN 'unknown' ELSE {HOURS_SQL.format(h="timestampdiff(HOUR, current_timestamp(), eta_ts)")} END AS eta,
  {AGE_SQL.format(m="heartbeat_age_minutes")} AS heartbeat_age,
  CASE WHEN heartbeat_at IS NULL OR run_state = 'complete' THEN '-'
       WHEN timeout_in_minutes < 0 THEN concat('overdue ', -timeout_in_minutes, 'm')
       ELSE concat('in ', timeout_in_minutes, 'm') END AS stale_in,
  CAST(status = 'DONE' AS INT) AS is_done,
  CAST(status IN ('ROW_CHECK', 'COL_CHECK', 'REPORTING') AS INT) AS is_running,
  CAST(status IN ('BLOCKED', 'STALLED', 'FAILED') AS INT) AS is_trouble,
  CAST(status = 'NOT_STARTED' AS INT) AS is_waiting,
  progress * coalesce(units_expected, 0) AS weighted_progress
FROM {schema}.v_pair_status"""
    daily = f"SELECT * FROM {schema}.v_pair_daily"
    blockers = f"""SELECT *, resolved_at IS NULL AS is_open,
  timestampdiff(HOUR, opened_at, coalesce(resolved_at, current_timestamp())) AS open_hours
FROM {schema}.dtrack_pair_blocker
WHERE resolved_at IS NULL OR resolved_at >= current_timestamp() - INTERVAL 30 DAYS"""
    return {"ds_status": status, "ds_daily": daily, "ds_blockers": blockers}


def dataset(name, display, sql):
    lines = sql.split("\n")
    return {"name": name, "displayName": display, "queryLines": [l + "\n" for l in lines[:-1]] + [lines[-1]]}


def frame(title, description=""):
    return {"showTitle": True, "title": title, "showDescription": bool(description), "description": description}


def tint_rules(field, tints):
    return {"type": "basic", "rules": [
        {"condition": {"operand": {"type": "data-value", "value": v}, "operator": "=", "fieldName": field},
         "backgroundColor": TINT[c][0], "foregroundColor": TINT[c][1]} for v, c in tints.items()]}


def column(field, title, kind="string", style=None, width=None):
    spec = {"fieldName": field, "displayName": title, "useForSearch": kind == "string"}
    if kind == "integer":
        spec["format"] = {"type": "number-plain", "abbreviation": "none", "decimalPlaces": {"type": "exact", "places": 0}}
        spec["contentAlignment"] = "right"
    elif kind == "percent":
        spec["format"] = {"type": "number-plain", "abbreviation": "none", "decimalPlaces": {"type": "exact", "places": 1}}
        spec["contentAlignment"] = "right"
    elif kind == "date":
        spec["format"] = {"type": "moment.js", "format": "YYYY-MM-DD"}
    elif kind == "datetime":
        spec["format"] = {"type": "moment.js", "format": "YYYY-MM-DD HH:mm"}
    if kind == "long":
        spec["actions"] = {"hover": {"type": "tooltip", "tooltipTemplate": "{{ @ }}"}}
    if style:
        spec["style"] = style
    if width:
        spec["columnWidth"] = width
    return spec


def table(name, ds, title, cols, orders, extra=(), filters=None, description="", rows=25):
    fields = [c["fieldName"] for c in cols] + [f for f in extra if f not in {c["fieldName"] for c in cols}]
    query = {"datasetName": ds, "fields": [{"name": f, "expression": f"`{f}`"} for f in fields], "disaggregated": True,
             "orders": [{"direction": d, "expression": f"`{f}`"} for f, d in orders]}
    if filters:
        query["filters"] = [{"expression": f} for f in filters]
    return {"name": name, "queries": [{"name": "main_query", "query": query}],
            "spec": {"version": 2, "widgetType": "table", "rowsPerPage": rows, "displayRowNumbers": False,
                     "encodings": {"columns": cols}, "data": {"queryName": "main_query"}, "frame": frame(title, description)}}


def counter(name, ds, title, agg, field, description=""):
    alias = f"{agg.lower()}({field})"
    return {"name": name, "queries": [{"name": "main_query", "query": {
        "datasetName": ds, "fields": [{"name": alias, "expression": f"{agg}(`{field}`)"}], "disaggregated": False}}],
        "spec": {"version": 2, "widgetType": "counter", "encodings": {"value": {"fieldName": alias, "displayName": title}},
                 "data": {"queryName": "main_query"}, "frame": frame(title, description)}}


def stacked_bar(name, ds, title, x, color, measure, tints):
    return {"name": name, "queries": [{"name": "main_query", "query": {"datasetName": ds, "fields": [
        {"name": x, "expression": f"`{x}`"}, {"name": color, "expression": f"`{color}`"},
        {"name": f"count({measure})", "expression": f"COUNT(`{measure}`)"}], "disaggregated": False}}],
        "spec": {"version": 3, "widgetType": "bar", "encodings": {
            "x": {"fieldName": x, "scale": {"type": "categorical"}, "displayName": x},
            "y": {"fieldName": f"count({measure})", "scale": {"type": "quantitative"}, "displayName": "Pairs"},
            "color": {"fieldName": color, "displayName": "Status", "scale": {"type": "categorical",
                      "mappings": [{"value": s, "color": SOLID[c]} for s, c in tints.items()]}}}, "frame": frame(title)}}


def line(name, ds, title, x, ys, color=None):
    fields = [{"name": x, "expression": f"`{x}`"}] + [{"name": f"sum({y})", "expression": f"SUM(`{y}`)"} for y in ys]
    enc = {"x": {"fieldName": x, "scale": {"type": "temporal"}, "displayName": x},
           "y": {"fieldName": f"sum({ys[0]})", "scale": {"type": "quantitative"}, "displayName": ys[0]}}
    if color:
        fields.append({"name": color, "expression": f"`{color}`"})
        enc["color"] = {"fieldName": color, "displayName": color, "scale": {"type": "categorical"}}
    return {"name": name, "queries": [{"name": "main_query", "query": {"datasetName": ds, "fields": fields, "disaggregated": False}}],
            "spec": {"version": 3, "widgetType": "line", "encodings": enc, "frame": frame(title)}}


def field_filter(name, kind, title, field, datasets):
    queries, fields = [], []
    for ds in datasets:
        qname = f"filter_{name}_{ds}"
        queries.append({"name": qname, "query": {"datasetName": ds, "fields": [
            {"name": field, "expression": f"`{field}`"},
            {"name": f"{field}_associativity", "expression": "COUNT_IF(`associative_filter_predicate_group`)"}], "disaggregated": False}})
        fields.append({"fieldName": field, "displayName": field, "queryName": qname})
    return {"name": name, "queries": queries, "spec": {"version": 2, "widgetType": kind, "encodings": {"fields": fields},
                                                       "frame": {"showTitle": True, "title": title}}}


def text(name, markdown):
    return {"name": name, "multilineTextboxSpec": {"lines": [l + "\n" for l in markdown.strip().split("\n")]}}


def at(widget, x, y, w, h):
    return {"widget": widget, "position": {"x": x, "y": y, "width": w, "height": h}}


def page(name, display, layout):
    return {"name": name, "displayName": display, "pageType": "PAGE_TYPE_CANVAS", "layoutVersion": "GRID_V1", "layout": layout}


def executive_page():
    cols = [
        column("display_name", "Pair", width=180),
        column("status_label", "Status", style=tint_rules("status", STATUS_TINTS), width=150),
        column("started_at", "Started", "datetime"),
        column("elapsed", "Elapsed", width=90),
        column("progress_pct", "Progress %", "percent"),
        column("eta", "ETA", width=100),
        column("target_done_date", "Target", "date"),
        column("schedule_health", "Schedule", style=tint_rules("schedule_health", HEALTH), width=110),
        column("heartbeat_age", "Heartbeat", width=100),
        column("stale_in", "Stale in", width=100),
        column("top_blocker", "Top blocker", "long", width=320),
    ]
    note = "**Status** per pair from the latest collector snapshot · **Progress** = 15% row check + 80% units compared + 5% report · **ETA** from the last 6 hours of unit throughput (unknown when idle)"
    layout = [
        at(text("exec_note", note), 0, 0, 8, 1),
        at(field_filter("f_wave", "filter-multi-select", "Wave", "wave", ["ds_status"]), 8, 0, 2, 1),
        at(field_filter("f_owner", "filter-multi-select", "Owner", "owner", ["ds_status"]), 10, 0, 2, 1),
        at(counter("k_total", "ds_status", "Pairs", "COUNT", "pair_name"), 0, 1, 2, 2),
        at(counter("k_done", "ds_status", "🟢 Done", "SUM", "is_done"), 2, 1, 2, 2),
        at(counter("k_running", "ds_status", "🔵 Running", "SUM", "is_running"), 4, 1, 2, 2),
        at(counter("k_trouble", "ds_status", "🔴 Needs attention", "SUM", "is_trouble", "Blocked, stalled or failed"), 6, 1, 2, 2),
        at(counter("k_waiting", "ds_status", "⚪ Not started", "SUM", "is_waiting"), 8, 1, 2, 2),
        at(counter("k_eta", "ds_status", "Last ETA", "MAX", "eta_ts", "Latest ETA across pairs"), 10, 1, 2, 2),
        at(table("exec_table", "ds_status", "Pair progress", cols, [("severity", "ASC"), ("priority", "ASC"), ("display_name", "ASC")],
                 extra=("status", "priority")), 0, 3, 12, 8),
        at(stacked_bar("exec_wave", "ds_status", "Status by wave", "wave", "status", "pair_name", STATUS_TINTS), 0, 11, 6, 5),
        at(line("exec_burnup", "ds_daily", "Burn-up: units done vs expected", "day", ["units_done", "units_expected"]), 6, 11, 6, 5),
    ]
    return page("executive", "Executive summary", layout)


def detailed_page():
    detail = [
        column("pair_name", "Pair", width=160), column("run_id", "Run", width=200), column("run_mode", "Mode"),
        column("run_state", "Run state"), column("stage", "Stage"),
        column("status_label", "Status", style=tint_rules("status", STATUS_TINTS), width=150),
        column("row_left_done_at", "Left count", "datetime"), column("row_done_at", "Row check", "datetime"),
        column("row_days_match", "Match days", "integer"), column("row_days_count_diff", "Count diff", "integer"),
        column("row_days_left_only", "Left only", "integer"), column("row_days_right_only", "Right only", "integer"),
        column("gate_policy", "Gate"), column("gate_passed", "Gate passed"), column("gate_excluded_ratio", "Excluded"),
        column("units_expected", "Expected", "integer"), column("units_published", "Published", "integer"),
        column("units_compared", "Compared", "integer"), column("units_dead", "Dead", "integer"), column("units_diff", "Diff", "integer"),
        column("units_per_hour", "Units/h"), column("eta_confidence", "ETA conf."), column("heartbeat_at", "Heartbeat", "datetime"),
        column("heartbeat_age", "Heartbeat age", width=110), column("heartbeat_deadline", "Stale at", "datetime"), column("stale_in", "Stale in", width=100),
        column("last_result_at", "Last result", "datetime"), column("run_root", "Run root", "long", width=300),
        column("collector_error", "Collector error", "long", width=240),
    ]
    blockers = [
        column("pair_name", "Pair", width=160), column("severity", "Sev", width=60), column("stage", "Stage"),
        column("category", "Category"), column("title", "Title", "long", width=320), column("source", "Source"),
        column("owner", "Owner"), column("opened_at", "Opened", "datetime"), column("open_hours", "Open (h)", "integer"),
        column("resolved_at", "Resolved", "datetime"), column("resolution", "Resolution", "long", width=240),
    ]
    layout = [
        at(field_filter("d_pair", "filter-multi-select", "Pair", "pair_name", ["ds_status", "ds_blockers", "ds_daily"]), 0, 0, 3, 1),
        at(field_filter("d_wave", "filter-multi-select", "Wave", "wave", ["ds_status"]), 3, 0, 2, 1),
        at(field_filter("d_status", "filter-multi-select", "Status", "status", ["ds_status"]), 5, 0, 2, 1),
        at(field_filter("d_day", "filter-date-range-picker", "History", "day", ["ds_daily"]), 7, 0, 3, 1),
        at(table("d_pairs", "ds_status", "Pair detail", detail, [("severity", "ASC"), ("pair_name", "ASC")], extra=("status",), rows=50), 0, 1, 12, 8),
        at(stacked_bar("d_row", "ds_status", "Row check days by pair", "pair_name", "status", "pair_name", STATUS_TINTS), 0, 9, 6, 5),
        at(line("d_history", "ds_daily", "Units done per pair", "day", ["units_done"], color="pair_name"), 6, 9, 6, 5),
        at(table("d_blockers", "ds_blockers", "Blockers (open first, resolved in the last 30 days)", blockers,
                 [("is_open", "DESC"), ("severity", "ASC"), ("opened_at", "DESC")], extra=("is_open",)), 0, 14, 12, 7),
    ]
    return page("detailed", "Detailed", layout)


def build(schema):
    sql = dataset_sql(schema)
    return {"datasets": [dataset("ds_status", "pair_status", sql["ds_status"]), dataset("ds_daily", "pair_daily", sql["ds_daily"]),
                         dataset("ds_blockers", "pair_blockers", sql["ds_blockers"])],
            "pages": [executive_page(), detailed_page()],
            "uiSettings": {"theme": {"widgetHeaderAlignment": "ALIGNMENT_UNSPECIFIED"}, "applyModeEnabled": False}}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--schema", default="main.dtrack")
    parser.add_argument("--out", default="pair_progress.lvdash.json")
    args = parser.parse_args()
    if not re.fullmatch(r"[A-Za-z_]\w*\.[A-Za-z_]\w*", args.schema):
        raise SystemExit("--schema must be catalog.schema")
    Path(args.out).write_text(json.dumps(build(args.schema), indent=2, ensure_ascii=False) + "\n")
    print(args.out)


if __name__ == "__main__":
    main()

In [ ]:
# Build the dashboard file, sanity-check its structure, and (EXECUTE) create or update it in the workspace.
assert subprocess.run([sys.executable, 'build_pair_progress_dashboard.py', '--schema', SCHEMA], capture_output=True, text=True, check=True)
dashboard = json.loads(Path('pair_progress.lvdash.json').read_text())
widgets = [item['widget']['name'] for page in dashboard['pages'] for item in page['layout']]
assert [p['name'] for p in dashboard['pages']] == ['executive', 'detailed'] and len(set(widgets)) == len(widgets)
assert all(f'{SCHEMA}.' in ''.join(ds['queryLines']) for ds in dashboard['datasets'])
for page in dashboard['pages']:
    for item in page['layout']:
        for query in item['widget'].get('queries', []):
            assert query['query']['datasetName'] in {ds['name'] for ds in dashboard['datasets']}
url = None
if EXECUTE:
    serialized = json.dumps(dashboard)
    mine = [d for d in w.lakeview.list() if d.display_name == 'dtrack pair progress']
    if mine:
        d = w.lakeview.update(mine[0].dashboard_id, serialized_dashboard=serialized)
    else:
        d = w.lakeview.create(display_name='dtrack pair progress', serialized_dashboard=serialized, parent_path=WORKSPACE_DIR, warehouse_id=WAREHOUSE_ID)
    w.lakeview.publish(d.dashboard_id, warehouse_id=WAREHOUSE_ID, embed_credentials=True)
    url = f'{w.config.host}/sql/dashboardsv3/{d.dashboard_id}'
{'file': str(OUT / 'pair_progress.lvdash.json'), 'datasets': [ds['name'] for ds in dashboard['datasets']], 'widgets': len(widgets), 'url': url}

In [ ]:
# Seed dtrack_pair_plan from the dtrack config (new pairs only; owners then fill wave / owner / target_done_date).
from dtrack_left.cli.config import load_config
plan = progress.plan_rows_from_config(load_config(CONFIG), updated_by=os.getenv('USER', ''))
merge = progress.merge_plan_sql(SCHEMA, plan)
Path('plan_seed.sql').write_text(merge + ';\n')
if EXECUTE and merge:
    run_sql(w, merge, WAREHOUSE_ID)
assert plan and 'WHEN NOT MATCHED' in merge
{'pairs': [r['pair_name'] for r in plan], 'generated': sorted(p.name for p in OUT.iterdir() if p.is_file())}